In [2]:
import pandas as pd
import sqlite3
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))
from config import DB_FILE

conn = sqlite3.connect(DB_FILE)
print("Connected")


Done!
Connected


I need to create an index so SQLite works faster. Without an index, it is like finding every mention of "Cost of Sales" on every single page of a book. With it, it jumps straight to the right pages.

In [3]:
conn.execute("CREATE INDEX IF NOT EXISTS idx_GL_account  ON GL (Account_key)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_GL_date     ON GL (Date)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_GL_territory ON GL (Territory_key)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_Calendar_date ON Calendar (Date)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_COA_account ON COA (Account_key)")
conn.commit()
print("Indexes created")

Indexes created


First we will connect GL to COA and Calendar and filter by Profit and Loss. I group the amounts by year and the full COA hierachy.

In [5]:
pl = pd.read_sql("""
    SELECT
        Calendar.Year,
        Calendar.Quarter,
        Calendar.Month,
        Class,
        SubClass,
        Account,
        SubAccount,
        SUM (Amount) AS Amount
    FROM gl
    JOIN coa      ON GL.Account_key = COA.Account_key
    JOIN calendar ON GL.Date = calendar.Date
    WHERE Report = 'Profit and Loss'
    GROUP BY
        Calendar.Year,
        Calendar.Quarter,
        Calendar.Month,
        Class,
        SubClass,
        Account
    ORDER BY
        Calendar.Year,
        Calendar.Quarter,
        Calendar.Month,
        Class
""", conn) 

print(f"Rows returned: {len(pl)}")
pl.head(10)

Rows returned: 580


,Year,Quarter,Month,Class,SubClass,Account,SubAccount,Amount
0,2018,Qtr 1,Feb,Interest & Tax,Interest Expense,Interest Expense,Interest Expense,-1320
1,2018,Qtr 1,Feb,Interest & Tax,Taxation,Taxation,Taxation,7422
2,2018,Qtr 1,Feb,Non-operating,Exchange Loss/Gain,Exchange Loss/Gain,Exchange Loss/Gain,64
3,2018,Qtr 1,Feb,Non-operating,Interest Income,Interest Income,Interest Income,1864
4,2018,Qtr 1,Feb,Operating account,Depreciation & Amortization,Amortization of Intangible Assets,Amortization of Intangible Assets,-1584
5,2018,Qtr 1,Feb,Operating account,Depreciation & Amortization,Equipment,Equipment,-32333
6,2018,Qtr 1,Feb,Operating account,Operating Expenses,Commissions,Commissions,-22058
7,2018,Qtr 1,Feb,Operating account,Operating Expenses,Entertainment,Entertainment,-467
8,2018,Qtr 1,Feb,Operating account,Operating Expenses,Office Supplies,Office Supplies,-8062
9,2018,Qtr 1,Feb,Operating account,Operating Expenses,Other Expenses,Other Expenses,-16892


We shall look for a positive amountof Sales. Cost of Sales of Sales and Expanses should be negative.

Now I simplify everything to create the classic P&L structure (Sales, Cost of Sales, Operating Expenses, etc., over my three-year period).

In [6]:
annual = pd.read_sql("""
    SELECT
        Calendar.Year,
        COA.Class,
        COA.SubClass,
        SUM(gl.Amount) AS Amount
    FROM GL
    JOIN COA       ON GL.Account_key = COA.Account_key
    JOIN Calendar  ON GL.Date        = Calendar.Date
    WHERE COA.Report = 'Profit and Loss'
    GROUP BY Calendar.Year, COA.Class, COA.SubClass
    ORDER BY Calendar.Year, COA.Class, COA.SubClass
""", conn)
display(annual)

,Year,Class,SubClass,Amount
0,2018,Interest & Tax,Interest Expense,-15840
1,2018,Interest & Tax,Taxation,-137582
2,2018,Non-operating,Dividend Income,15917
3,2018,Non-operating,Exchange Loss/Gain,2214
4,2018,Non-operating,Gain/Loss on Sales of Asset,4850
5,2018,Non-operating,Interest Income,13496
6,2018,Operating account,Depreciation & Amortization,-407004
7,2018,Operating account,Operating Expenses,-1235441
8,2018,Trading account,Cost of Sales,-1192182
9,2018,Trading account,Sales,3575428


For each class, subclass and year we can now see the different cash flows

The actual P&L Statement: I take the annual summary (from above) to compute it with the key P&L subtotals (Gross Profit, Operating Profit and Net Profit - for each year)

In [7]:
waterfall = annual.groupby(['Year', 'SubClass'])['Amount'].sum().unstack('SubClass').fillna(0)

# Calculate P&L lines
waterfall['Revenue']          = waterfall.get('Sales', 0)
waterfall['Gross Profit']     = waterfall['Revenue'] + waterfall.get('Cost of Sales', 0)
waterfall['EBIT']             = waterfall['Gross Profit'] + waterfall.get('Operating Expenses', 0) + waterfall.get('Depreciation & Amortization', 0)
waterfall['Net Profit']       = waterfall['EBIT'] + waterfall.get('Interest Income', 0) + waterfall.get('Interest Expense', 0) + waterfall.get('Taxation', 0)

# Show only the key lines
summary = waterfall[['Revenue', 'Gross Profit', 'EBIT', 'Net Profit']]
print(summary)


SubClass  Revenue  Gross Profit     EBIT  Net Profit
Year                                                
2018      3575428       2383246   740801      600875
2019      5697845       3968546  1475688     1272606
2020      7835369       5341360  1522166     1248548


Gross Profit should be less than Revenue ---> It is.<br>
EBIT should be less than Gross Profit ---> It is. <br>
Net Profit should be the smallest number ---> It is. <br>

In [8]:
#Just as quick check:

print(annual['SubClass'].unique())
print(annual['Class'].unique())

['Interest Expense' 'Taxation' 'Dividend Income' 'Exchange Loss/Gain'
 'Gain/Loss on Sales of Asset' 'Interest Income'
 'Depreciation & Amortization' 'Operating Expenses' 'Cost of Sales'
 'Sales']
['Interest & Tax' 'Non-operating' 'Operating account' 'Trading account']


I reshape the Data and pivot eit so each Subclass becomes its own column.

In [9]:
waterfall = annual.groupby(['Year', 'SubClass'])['Amount'].sum().unstack('SubClass').fillna(0)

print(waterfall.shape)
display(waterfall)

(3, 10)


SubClass,Cost of Sales,Depreciation & Amortization,Dividend Income,Exchange Loss/Gain,Gain/Loss on Sales of Asset,Interest Expense,Interest Income,Operating Expenses,Sales,Taxation
Year,,,,,,,,,,
2018,-1192182,-407004,15917,2214,4850,-15840,13496,-1235441,3575428,-137582
2019,-1729299,-532056,21751,3705,5085,-21600,16621,-1960802,5697845,-198103
2020,-2494009,-714840,30030,5016,6351,-25404,30868,-3104354,7835369,-279082


Now we use the pivoted data to calculate the 5 key P&L subtotals

In [10]:
#Revenue
waterfall['1. Revenue']      = waterfall['Sales']

#Gross Profit = Revenue + Cost of Sales (CoS is already negative)
waterfall['2. Gross Profit'] = waterfall['1. Revenue'] + waterfall['Cost of Sales']

#EBIT = Gross Profit - Operating Expenses - Depreciation
waterfall['3. EBIT']         = (waterfall['2. Gross Profit']
                                + waterfall['Operating Expenses']
                                + waterfall['Depreciation & Amortization'])

#EBT = EBIT + all Non-operating items
waterfall['4. EBT']          = (waterfall['3. EBIT']
                                + waterfall['Interest Income']
                                + waterfall['Interest Expense']
                                + waterfall['Dividend Income']
                                + waterfall['Exchange Loss/Gain']
                                + waterfall['Gain/Loss on Sales of Asset'])

#Net Profit = EBT - Taxation
waterfall['5. Net Profit']   = waterfall['4. EBT'] + waterfall['Taxation']

#To display only the summary lines
summary = waterfall[['1. Revenue', '2. Gross Profit', '3. EBIT', '4. EBT', '5. Net Profit']]
display(summary)


SubClass,1. Revenue,2. Gross Profit,3. EBIT,4. EBT,5. Net Profit
Year,,,,,
2018,3575428,2383246,740801,761438,623856
2019,5697845,3968546,1475688,1501250,1303147
2020,7835369,5341360,1522166,1569027,1289945


Each line should be smaller than the one above -> money gets deducted at every stage ---> It is. <br> 
All three years should show positive Net Profit for a healthy business ---> It shows.<br> 
A growing trend from 2018 → 2020 is a good sign <br> 
If any line unexpectedly jumps higher than the one above it, one of the signs is wrong <br> 
<br> 
<br> 

Now add Margin Percentages - the absolute numbers matter less than the margins, and whether they are improving year on year.

In [12]:
margins = pd.DataFrame(index=summary.index)

margins['Revenue'] = summary['1. Revenue']
margins['Gross Margin %'] = (summary['2. Gross Profit'] / summary['1. Revenue'] * 100).round(1)
margins['EBIT Margin %'] = (summary['3. EBIT'] / summary['1. Revenue'] * 100).round(1)
margins['EBT Margin %'] = (summary['4. EBT']  / summary['1. Revenue'] * 100).round(1)
margins['Net Margin %'] = (summary['5. Net Profit'] / summary['1. Revenue'] * 100).round(1)

display(margins)

,Revenue,Gross Margin %,EBIT Margin %,EBT Margin %,Net Margin %
Year,,,,,
2018,3575428,66.7,20.7,21.3,17.4
2019,5697845,69.6,25.9,26.3,22.9
2020,7835369,68.2,19.4,20.0,16.5
